In [ ]:
!apt-get -qq update
!apt-get -qq install -y openjdk-21-jre-headless libomp-dev

# Force java to 21 (important)
!update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java
!java -version

!pip -q install -U huggingface_hub hf_transfer transformers pyserini

# FAISS GPU for CUDA 12.x
!pip -q install -U faiss-gpu-cu12


In [1]:
# ============================================================
# Hybrid Retrieval: BM25 (Pyserini) + DPR FAISS + BGE Reranker
# Colab A100-friendly, downloads indices from Hugging Face.
#
# IMPORTANT:
# - Java 21 MUST be active before importing pyserini.
# - If you already imported pyserini/jnius earlier in this runtime, restart runtime.
# ============================================================

import os
import subprocess

# -----------------------------
# Force Java 21 BEFORE pyserini import
# -----------------------------
JAVA_HOME = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["JDK_HOME"] = JAVA_HOME
os.environ["PATH"] = os.path.join(JAVA_HOME, "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = os.path.join(JAVA_HOME, "lib", "server") + os.pathsep + os.environ.get("LD_LIBRARY_PATH", "")

# Verify Java version (should show 21)
print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode())

# -----------------------------
# Imports (pyserini AFTER Java env)
# -----------------------------
import json
import numpy as np
import faiss
import torch

from huggingface_hub import snapshot_download, hf_hub_download
from pyserini.search.lucene import LuceneSearcher

from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizerFast,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

# ============================================================
# CONFIG
# ============================================================

# --- HF repos ---
DPR_REPO  = "MinaGabriel/wikipedia-18-en-dpr-faiss-100tok"
BM25_REPO = "MinaGabriel/wiki18-bm25-index"

# --- Models ---
DPR_MODEL         = "facebook/dpr-question_encoder-single-nq-base"
RERANK_MODEL_NAME = "BAAI/bge-reranker-base"

# --- Retrieval params ---
K_BM25  = 100
K_DENSE = 30
FUSE_K  = 200
TOP_K   = 30

DPR_TEXT_CHARS  = 2000
BM25_TEXT_CHARS = 2000

# --- GPUs (Colab usually has 1 GPU; use 0 for both) ---
DENSE_GPU  = 0
RERANK_GPU = 0

# Optional: faster HF downloads (needs hf_transfer installed)
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

# ============================================================
# DEVICES
# ============================================================
use_cuda = torch.cuda.is_available()
device_dense = torch.device(f"cuda:{DENSE_GPU}" if use_cuda else "cpu")
device_rerank = torch.device(f"cuda:{RERANK_GPU}" if use_cuda else "cpu")

print(f"Dense retrieval device:  {device_dense}")
print(f"Reranker device:         {device_rerank}")
print(f"GPU count: {torch.cuda.device_count()}")

# ============================================================
# DOWNLOAD DPR FAISS + META (only those two files)
# ============================================================
print("Downloading DPR FAISS + META from Hugging Face...")

INDEX_PATH = hf_hub_download(
    repo_id=DPR_REPO,
    repo_type="dataset",
    filename="index/wiki_dpr.index",
)
META_PATH = hf_hub_download(
    repo_id=DPR_REPO,
    repo_type="dataset",
    filename="index/wiki_dpr_meta.npz",
)

print("INDEX_PATH:", INDEX_PATH)
print("META_PATH: ", META_PATH)

# ============================================================
# LOAD BM25
# ============================================================
print("Loading BM25 index...")
bm25_dir = snapshot_download(repo_id=BM25_REPO, repo_type="dataset")
index_dir = os.path.join(bm25_dir, "bm25_index")
searcher = LuceneSearcher(index_dir)

# ============================================================
# LOAD DPR FAISS + META
# ============================================================
print("Loading FAISS index...")
index_cpu = faiss.read_index(INDEX_PATH)

index = index_cpu
if use_cuda:
    try:
        print("Moving FAISS index to GPU...")
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, DENSE_GPU, index_cpu)
        del index_cpu
        print("FAISS is on GPU.")
    except Exception as e:
        print("GPU FAISS move failed; using CPU FAISS:", repr(e))
        index = index_cpu

print("Loading META npz...")
# With allow_pickle/object arrays, mmap_mode usually won't help; keep it simple.
meta = np.load(META_PATH, allow_pickle=True)
doc_ids = meta["doc_ids"]
titles  = meta["titles"]
texts   = meta["texts"]

print(f"Meta loaded: {len(doc_ids):,} docs")

# ============================================================
# LOAD DPR QUESTION ENCODER
# ============================================================
print("Loading DPR model...")
q_tok = DPRQuestionEncoderTokenizerFast.from_pretrained(DPR_MODEL)
q_model = DPRQuestionEncoder.from_pretrained(DPR_MODEL).to(device_dense).eval()

# ============================================================
# LOAD BGE RERANKER
# ============================================================
print("Loading BGE reranker...")
rerank_tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL_NAME)
rerank_model = AutoModelForSequenceClassification.from_pretrained(
    RERANK_MODEL_NAME,
    torch_dtype=torch.float16 if device_rerank.type == "cuda" else torch.float32,
    attn_implementation="eager",
).to(device_rerank).eval()

# ============================================================
# HELPERS
# ============================================================
def _norm_key(title: str, text: str) -> str:
    t = (title or "").strip().lower()
    x = (text or "").strip().lower()
    return t + "||" + x[:200]

# ============================================================
# DPR DENSE SEARCH
# ============================================================
@torch.no_grad()
def dense_search(question: str, top_k: int = 30):
    enc = q_tok(question, truncation=True, padding=True, max_length=64, return_tensors="pt")
    enc = {k: v.to(device_dense) for k, v in enc.items()}
    emb = q_model(**enc).pooler_output  # [1, d]

    vec = np.ascontiguousarray(emb.detach().cpu().numpy().astype(np.float32))

    # Keep this if index was built for cosine/IP with normalized vectors.
    faiss.normalize_L2(vec)

    scores, idxs = index.search(vec, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], idxs[0]), start=1):
        idx = int(idx)
        if idx < 0:
            continue
        results.append({
            "source": "dpr",
            "rank": rank,
            "score": float(score),
            "doc_id": str(doc_ids[idx]),
            "title": str(titles[idx]),
            "text": str(texts[idx])[:DPR_TEXT_CHARS],
        })
    return results

# ============================================================
# BM25 SEARCH
# ============================================================
def bm25_search(question: str, top_k: int = 100):
    hits = searcher.search(question, top_k)
    results = []
    for rank, h in enumerate(hits, start=1):
        raw_json = searcher.doc(h.docid).raw()
        record = json.loads(raw_json)
        contents = record.get("contents", "") or ""
        results.append({
            "source": "bm25",
            "rank": rank,
            "score": float(h.score),
            "doc_id": str(h.docid),
            "title": record.get("title", ""),
            "text": contents[:BM25_TEXT_CHARS],
        })
    return results

# ============================================================
# RRF FUSION + DEDUPE
# ============================================================
def fuse_candidates(bm25_res, dpr_res, fuse_k: int = 200, rrf_k: int = 60):
    pool = {}
    for r in (bm25_res or []) + (dpr_res or []):
        key = _norm_key(r.get("title", ""), r.get("text", ""))
        if key not in pool:
            pool[key] = {**r, "rrf": 0.0}
        pool[key]["rrf"] += 1.0 / (rrf_k + int(r["rank"]))

    fused = sorted(pool.values(), key=lambda x: x["rrf"], reverse=True)
    return fused[:fuse_k]

# ============================================================
# BGE RERANK (CROSS-ENCODER)
# ============================================================
@torch.no_grad()
def rerank_with_bge_reranker(
    question: str,
    candidates: list,
    top_k: int = 25,
    batch_size: int = 16,
    max_length: int = 384,
    text_key: str = "text",
):
    if not candidates:
        return []

    passages = [(c.get(text_key) or "").strip() for c in candidates]
    pairs = [(question, p) for p in passages]

    scores = np.empty(len(pairs), dtype=np.float32)

    for start in range(0, len(pairs), batch_size):
        batch_pairs = pairs[start:start + batch_size]
        enc = rerank_tokenizer(
            batch_pairs,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc = {k: v.to(device_rerank) for k, v in enc.items()}

        out = rerank_model(**enc)
        batch_scores = out.logits.squeeze(-1)
        scores[start:start + len(batch_pairs)] = batch_scores.detach().float().cpu().numpy()

    order = np.argsort(-scores)[:top_k]
    reranked = []
    for i in order:
        item = dict(candidates[int(i)])
        item["rerank_score"] = float(scores[int(i)])
        reranked.append(item)

    return reranked

# ============================================================
# MAIN PIPELINE
# ============================================================
def retrieve_and_rerank(
    question: str,
    k_bm25: int = K_BM25,
    k_dense: int = K_DENSE,
    fuse_k: int = FUSE_K,
    top_k: int = TOP_K,
):
    bm25_res = bm25_search(question, top_k=k_bm25)
    dpr_res  = dense_search(question, top_k=k_dense)
    fused    = fuse_candidates(bm25_res, dpr_res, fuse_k=fuse_k)

    reranked = rerank_with_bge_reranker(
        question=question,
        candidates=fused,
        top_k=top_k,
        batch_size=16,
        max_length=384,
        text_key="text",
    )
    return reranked



openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-124.04, mixed mode, sharing)



Dense retrieval device:  cuda:0
Reranker device:         cuda:0
GPU count: 1
INDEX_PATH: /home/mina/hf_cache/hub/datasets--MinaGabriel--wikipedia-18-en-dpr-faiss-100tok/snapshots/1e3c1069ba491fcfcb1844809f61bedbf6ce28b1/index/wiki_dpr.index
META_PATH:  /home/mina/hf_cache/hub/datasets--MinaGabriel--wikipedia-18-en-dpr-faiss-100tok/snapshots/1e3c1069ba491fcfcb1844809f61bedbf6ce28b1/index/wiki_dpr_meta.npz
Loading BM25 index...


Fetching 190 files:   0%|          | 0/190 [00:00<?, ?it/s]

Mar 12, 2026 7:35:34 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


Loading FAISS index...
Moving FAISS index to GPU...
FAISS is on GPU.
Loading META npz...
Meta loaded: 6,407,814 docs
Loading DPR model...


...

DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-single-nq-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading BGE reranker...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# SMOKE TEST

from prettytable import PrettyTable

def display_retrieval_results(results, max_text_len=80):
    table = PrettyTable()
    table.field_names = ["Rank", "Source", "Doc ID", "RRF", "BM25/Dense", "Rerank", "Text Snippet"]
    
    # Column alignment
    table.align["Rank"]       = "c"
    table.align["Source"]     = "c"
    table.align["Doc ID"]     = "c"
    table.align["RRF"]        = "r"
    table.align["BM25/Dense"] = "r"
    table.align["Rerank"]     = "r"
    table.align["Text Snippet"] = "l"
    
    table.max_width["Text Snippet"] = max_text_len
    table.hrules = 1  # line between every row

    # Sort by rerank_score descending
    sorted_results = sorted(results, key=lambda x: x.get("rerank_score", 0), reverse=True)

    for i, doc in enumerate(sorted_results, 1):
        snippet = doc.get("text", "").replace("\n", " ").strip()
        snippet = (snippet[:]) if len(snippet) > max_text_len else snippet

        table.add_row([
            i,
            doc.get("source", "").upper(),
            doc.get("doc_id", ""),
            f'{doc.get("rrf", 0):.5f}',
            f'{doc.get("score", 0):.3f}',
            f'{doc.get("rerank_score", 0):.4f}',
            snippet,
        ])

    print(f"\n{'='*40}")
    print(f"  Query Results — {len(results)} docs retrieved")
    print(f"{'='*40}\n")
    print(table)


# ── Smoke test ──────────────────────────────
q = "In what city was John XIX born?"  # Example query; change as needed
out = retrieve_and_rerank(q, top_k=20)
display_retrieval_results(out)


  Query Results — 20 docs retrieved

+------+--------+----------+---------+------------+---------+----------------------------------------------------------------------------------+
| Rank | Source |  Doc ID  |     RRF | BM25/Dense |  Rerank | Text Snippet                                                                     |
+------+--------+----------+---------+------------+---------+----------------------------------------------------------------------------------+
|  1   |  DPR   | 3799017  | 0.01190 |      0.584 |  3.6133 | Pope John XIX of Alexandria (Abba Youannis XIX), 113th Pope of Alexandria &      |
|      |        |          |         |            |         | Patriarch of the See of St. Mark.  A monk He joined the Paromeos Monastery in    |
|      |        |          |         |            |         | the Nitrian Desert as a monk and was sent to Greece to study Theology.           |
|      |        |          |         |            |         | Afterwards, Pope Cyril V appoi

In [ ]:
import re, numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset

# Bucket thresholds
LT_MIN, LT_MAX   = 10, 100
INF_MIN, INF_MAX = 100, 10_000
FREQ_MIN         = 10_000

_WS    = re.compile(r"\s+")
_PUNCT = re.compile(r"[^\w\s]")

def norm(s):
    s = "" if s is None else str(s).lower()
    s = _PUNCT.sub(" ", s)
    return _WS.sub(" ", s).strip()

def parse_answers(x):
    if isinstance(x, list): return [str(a) for a in x]
    if isinstance(x, str):
        import json
        try: v = json.loads(x); return [str(a) for a in v] if isinstance(v, list) else [x]
        except: return [x]
    return []

def hit_rank(docs, gold, topk):
    gold_norm = [norm(g) for g in gold if norm(g)]
    for i, d in enumerate(docs[:topk], 1):
        txt = norm(d.get("text", "") if isinstance(d, dict) else str(d))
        if any(g in txt for g in gold_norm):
            return i
    return None

# Load dataset
ds = load_dataset("MinaGabriel/popqa-retrieval20-with-sre")["test"]

# Accumulators
results = {
    "long-tail":  {"hits10": [], "hits20": []},
    "infrequent": {"hits10": [], "hits20": []},
    "frequent":   {"hits10": [], "hits20": []},
}

def get_bucket(freq):
    try: freq = int(freq)
    except: return None
    if LT_MIN  <= freq < LT_MAX:  return "long-tail"
    if INF_MIN <= freq < INF_MAX: return "infrequent"
    if freq >= FREQ_MIN:          return "frequent"
    return None

for ex in tqdm(ds, desc="DPR-only"):
    bucket = get_bucket(ex["s_pop"])
    if bucket is None:
        continue

    retrieved = dense_search(ex["question"], top_k=30)
    gold      = parse_answers(ex["possible_answers"])
    r         = hit_rank(retrieved, gold, topk=20)

    results[bucket]["hits10"].append(1 if (r is not None and r <= 10) else 0)
    results[bucket]["hits20"].append(1 if (r is not None and r <= 20) else 0)

# Print results
print("\n=== DPR-Only PopQA Results ===")
for b in ["long-tail", "infrequent", "frequent"]:
    h = results[b]
    r10 = 100 * np.mean(h["hits10"])
    r20 = 100 * np.mean(h["hits20"])
    print(f"{b:<12}  R@10={r10:.1f}  R@20={r20:.1f}  (n={len(h['hits10'])})")

In [ ]:
import re, numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset

# Bucket thresholds
LT_MIN, LT_MAX   = 10, 100
INF_MIN, INF_MAX = 100, 10_000
FREQ_MIN         = 10_000

_WS    = re.compile(r"\s+")
_PUNCT = re.compile(r"[^\w\s]")

def norm(s):
    s = "" if s is None else str(s).lower()
    s = _PUNCT.sub(" ", s)
    return _WS.sub(" ", s).strip()

def parse_answers(x):
    if isinstance(x, list): return [str(a) for a in x]
    if isinstance(x, str):
        import json
        try: v = json.loads(x); return [str(a) for a in v] if isinstance(v, list) else [x]
        except: return [x]
    return []

def hit_rank(docs, gold, topk):
    gold_norm = [norm(g) for g in gold if norm(g)]
    for i, d in enumerate(docs[:topk], 1):
        txt = norm(d.get("text", "") if isinstance(d, dict) else str(d))
        if any(g in txt for g in gold_norm):
            return i
    return None

# Load dataset
ds = load_dataset("MinaGabriel/entityquestions-retrieval20-with-sre")["train"]

# Accumulators
results = {
    "long-tail":  {"hits10": [], "hits20": []},
    "infrequent": {"hits10": [], "hits20": []},
    "frequent":   {"hits10": [], "hits20": []},
}

def get_bucket(freq):
    try: freq = int(freq)
    except: return None
    if LT_MIN  <= freq < LT_MAX:  return "long-tail"
    if INF_MIN <= freq < INF_MAX: return "infrequent"
    if freq >= FREQ_MIN:          return "frequent"
    return None

for ex in tqdm(ds, desc="DPR-only"):
    bucket = get_bucket(ex["s_pop_avg"])
    if bucket is None:
        continue

    retrieved = dense_search(ex["question"], top_k=30)
    gold      = parse_answers(ex["possible_answers"])
    r         = hit_rank(retrieved, gold, topk=20)

    results[bucket]["hits10"].append(1 if (r is not None and r <= 10) else 0)
    results[bucket]["hits20"].append(1 if (r is not None and r <= 20) else 0)

# Print results
print("\n=== DPR-Only EntityQuestions Results ===")
for b in ["long-tail", "infrequent", "frequent"]:
    h = results[b]
    r10 = 100 * np.mean(h["hits10"])
    r20 = 100 * np.mean(h["hits20"])
    print(f"{b:<12}  R@10={r10:.1f}  R@20={r20:.1f}  (n={len(h['hits10'])})")

### POPQA Hybrid Retrieval


In [ ]:
import re, numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset

# Bucket thresholds
LT_MIN, LT_MAX   = 10, 100
INF_MIN, INF_MAX = 100, 10_000
FREQ_MIN         = 10_000

_WS    = re.compile(r"\s+")
_PUNCT = re.compile(r"[^\w\s]")

def norm(s):
    s = "" if s is None else str(s).lower()
    s = _PUNCT.sub(" ", s)
    return _WS.sub(" ", s).strip()

def parse_answers(x):
    if isinstance(x, list): return [str(a) for a in x]
    if isinstance(x, str):
        import json
        try:
            v = json.loads(x)
            return [str(a) for a in v] if isinstance(v, list) else [x]
        except:
            return [x]
    return []

def hit_rank(docs, gold, topk):
    gold_norm = [norm(g) for g in gold if norm(g)]
    for i, d in enumerate(docs[:topk], 1):
        txt = norm(d.get("text", "") if isinstance(d, dict) else str(d))
        if any(g in txt for g in gold_norm):
            return i
    return None


# ============================================================
# Load dataset
# ============================================================

ds = load_dataset("MinaGabriel/popqa-retrieval20-with-sre")["test"]


# ============================================================
# Accumulators
# ============================================================

results = {
    "long-tail":  {"hits10": [], "hits20": []},
    "infrequent": {"hits10": [], "hits20": []},
    "frequent":   {"hits10": [], "hits20": []},
}


# ============================================================
# Bucket assignment
# ============================================================

def get_bucket(freq):
    try: freq = int(freq)
    except: return None
    if LT_MIN  <= freq < LT_MAX:  return "long-tail"
    if INF_MIN <= freq < INF_MAX: return "infrequent"
    if freq >= FREQ_MIN:          return "frequent"
    return None


# ============================================================
# Hybrid Retrieval Evaluation
# ============================================================

for ex in tqdm(ds, desc="Hybrid Retrieval"):

    bucket = get_bucket(ex["s_pop"])
    if bucket is None:
        continue

    # Step 1: BM25
    bm25_res = bm25_search(ex["question"], top_k=100)

    # Step 2: DPR
    dpr_res = dense_search(ex["question"], top_k=30)

    # Step 3: RRF fusion
    retrieved = fuse_candidates(bm25_res, dpr_res, fuse_k=200)

    gold = parse_answers(ex["possible_answers"])
    r    = hit_rank(retrieved, gold, topk=20)

    results[bucket]["hits10"].append(1 if (r is not None and r <= 10) else 0)
    results[bucket]["hits20"].append(1 if (r is not None and r <= 20) else 0)


# ============================================================
# Print results
# ============================================================

print("\n=== Hybrid Retrieval (BM25+DPR RRF) PopQA Results ===")

for b in ["long-tail", "infrequent", "frequent"]:
    h = results[b]
    r10 = 100 * np.mean(h["hits10"])
    r20 = 100 * np.mean(h["hits20"])
    print(f"{b:<12}  R@10={r10:.1f}  R@20={r20:.1f}  (n={len(h['hits10'])})")

In [ ]:
import re, numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset

# Bucket thresholds
LT_MIN, LT_MAX   = 10, 100
INF_MIN, INF_MAX = 100, 10_000
FREQ_MIN         = 10_000

_WS    = re.compile(r"\s+")
_PUNCT = re.compile(r"[^\w\s]")

def norm(s):
    s = "" if s is None else str(s).lower()
    s = _PUNCT.sub(" ", s)
    return _WS.sub(" ", s).strip()

def parse_answers(x):
    if isinstance(x, list): return [str(a) for a in x]
    if isinstance(x, str):
        import json
        try:
            v = json.loads(x)
            return [str(a) for a in v] if isinstance(v, list) else [x]
        except:
            return [x]
    return []

def hit_rank(docs, gold, topk):
    gold_norm = [norm(g) for g in gold if norm(g)]
    for i, d in enumerate(docs[:topk], 1):
        txt = norm(d.get("text", "") if isinstance(d, dict) else str(d))
        if any(g in txt for g in gold_norm):
            return i
    return None


# ============================================================
# Load dataset
# ============================================================

ds = load_dataset("MinaGabriel/entityquestions-retrieval20-with-sre")["train"]


# ============================================================
# Accumulators
# ============================================================

results = {
    "long-tail":  {"hits10": [], "hits20": []},
    "infrequent": {"hits10": [], "hits20": []},
    "frequent":   {"hits10": [], "hits20": []},
}


# ============================================================
# Bucket assignment
# ============================================================

def get_bucket(freq):
    try: freq = int(freq)
    except: return None
    if LT_MIN  <= freq < LT_MAX:  return "long-tail"
    if INF_MIN <= freq < INF_MAX: return "infrequent"
    if freq >= FREQ_MIN:          return "frequent"
    return None


# ============================================================
# Hybrid Retrieval Evaluation
# ============================================================

for ex in tqdm(ds, desc="Hybrid Retrieval"):

    bucket = get_bucket(ex["s_pop"])
    if bucket is None:
        continue

    # Step 1: BM25
    bm25_res = bm25_search(ex["question"], top_k=100)

    # Step 2: DPR
    dpr_res = dense_search(ex["question"], top_k=30)

    # Step 3: RRF fusion
    retrieved = fuse_candidates(bm25_res, dpr_res, fuse_k=200)

    gold = parse_answers(ex["possible_answers"])
    r    = hit_rank(retrieved, gold, topk=20)

    results[bucket]["hits10"].append(1 if (r is not None and r <= 10) else 0)
    results[bucket]["hits20"].append(1 if (r is not None and r <= 20) else 0)


# ============================================================
# Print results
# ============================================================

print("\n=== Hybrid Retrieval (BM25+DPR RRF) PopQA Results ===")

for b in ["long-tail", "infrequent", "frequent"]:
    h = results[b]
    r10 = 100 * np.mean(h["hits10"])
    r20 = 100 * np.mean(h["hits20"])
    print(f"{b:<12}  R@10={r10:.1f}  R@20={r20:.1f}  (n={len(h['hits10'])})")

In [ ]:
# fuseqa_retrieval_rank_report_tiers.py
# Reproduce FUSE-QA (+Reranker) numbers

import json
import re
import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset

# ============================================================
# SETTINGS (MATCH PAPER)
# ============================================================

OUT_FILE = "fuseqa_popqa_retrieval_rank_report_tiers.txt"

K_BM25 = 100
K_DENSE = 30
FUSE_K = 200
TOP_K = 20
K_LIST = [1, 5, 10, 20]

LT_THRESHOLD = 100
FREQ_THRESHOLD = 10000

TEXT_KEY = "text"
ANS_KEY = "possible_answers"

# ============================================================
# HELPERS
# ============================================================

_WS = re.compile(r"\s+")
_PUNCT = re.compile(r"[^\w\s]")

def norm(s):
    s = "" if s is None else str(s).lower()
    s = _PUNCT.sub(" ", s)
    return _WS.sub(" ", s).strip()

def parse_answers(x):

    if x is None:
        return []

    if isinstance(x, list):
        return [str(a) for a in x]

    if isinstance(x, str):
        s = x.strip()

        if s.startswith("[") and s.endswith("]"):
            try:
                v = json.loads(s)
                if isinstance(v, list):
                    return [str(a) for a in v]
            except:
                pass

        return [s]

    return [str(x)]

def first_hit_rank(retrieved_docs, gold_aliases):

    if not retrieved_docs or not gold_aliases:
        return None

    gold_norm = [norm(g) for g in gold_aliases if norm(g)]

    for i, d in enumerate(retrieved_docs[:TOP_K], start=1):

        txt = d.get(TEXT_KEY, "") if isinstance(d, dict) else str(d)
        txt = norm(txt)

        for g in gold_norm:
            if g and g in txt:
                return i

    return None

def mean01(xs):
    return float(np.mean(xs)) if xs else 0.0


# ============================================================
# LOAD DATASET
# ============================================================

ds = load_dataset("MinaGabriel/popqa-retrieval20-with-sre")["test"]
n = len(ds)

print("Dataset size:", n)


# ============================================================
# METRICS
# ============================================================

metrics = {
    "ALL": {"hits": {k: [] for k in K_LIST}, "n": 0},
    "LONG-TAIL": {"hits": {k: [] for k in K_LIST}, "n": 0},
    "INFREQUENT": {"hits": {k: [] for k in K_LIST}, "n": 0},
    "FREQUENT": {"hits": {k: [] for k in K_LIST}, "n": 0},
}


# ============================================================
# EVALUATION LOOP
# ============================================================

for ex in tqdm(ds, desc="FUSE-QA Evaluation"):

    q = ex["question"]
    gold = parse_answers(ex.get(ANS_KEY))
    s_pop = int(ex.get("s_pop", 0))

    # ------------------------------------------------
    # FULL FUSE-QA PIPELINE
    # BM25 + DPR + RRF + BGE reranker
    # ------------------------------------------------

    retrieved = retrieve_and_rerank(
        q,
        k_bm25=K_BM25,
        k_dense=K_DENSE,
        fuse_k=FUSE_K,
        top_k=TOP_K
    )

    hit_rank = first_hit_rank(retrieved, gold)

    # ------------------------------------------------
    # Tier assignment
    # ------------------------------------------------

    if s_pop < LT_THRESHOLD:
        tier = "LONG-TAIL"
    elif s_pop < FREQ_THRESHOLD:
        tier = "INFREQUENT"
    else:
        tier = "FREQUENT"

    for group in ["ALL", tier]:

        metrics[group]["n"] += 1

        for k in K_LIST:

            val = 1 if (hit_rank is not None and hit_rank <= k) else 0
            metrics[group]["hits"][k].append(val)


# ============================================================
# REPORT GENERATION
# ============================================================

lines = []
lines.append("=" * 80)
lines.append("FUSE-QA Retrieval Report (BM25 + DPR + RRF + BGE)")
lines.append("=" * 80)
lines.append(f"Dataset: PopQA test")
lines.append(f"n = {n}")
lines.append("")
lines.append("Tier Definitions:")
lines.append(f"LONG-TAIL   s_pop < {LT_THRESHOLD}")
lines.append(f"INFREQUENT  {LT_THRESHOLD} ≤ s_pop < {FREQ_THRESHOLD}")
lines.append(f"FREQUENT    s_pop ≥ {FREQ_THRESHOLD}")
lines.append("")

for name in ["ALL", "LONG-TAIL", "INFREQUENT", "FREQUENT"]:

    m = metrics[name]

    lines.append(f"{name:<12} (n={m['n']})")

    for k in K_LIST:
        score = mean01(m["hits"][k])
        lines.append(f"  R@{k:<2} = {score:.4f}")

    lines.append("")


with open(OUT_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("Saved report:", OUT_FILE)

In [ ]:
# fuseqa_retrieval_rank_report_tiers.py
# Reproduce FUSE-QA (+Reranker) numbers

import json
import re
import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset

# ============================================================
# SETTINGS (MATCH PAPER)
# ============================================================

OUT_FILE = "fuseqa_entityquestions_retrieval_rank_report_tiers.txt"

K_BM25 = 100
K_DENSE = 30
FUSE_K = 200
TOP_K = 20
K_LIST = [1, 5, 10, 20]

LT_THRESHOLD = 100
FREQ_THRESHOLD = 10000

TEXT_KEY = "text"
ANS_KEY = "possible_answers"

# ============================================================
# HELPERS
# ============================================================

_WS = re.compile(r"\s+")
_PUNCT = re.compile(r"[^\w\s]")

def norm(s):
    s = "" if s is None else str(s).lower()
    s = _PUNCT.sub(" ", s)
    return _WS.sub(" ", s).strip()

def parse_answers(x):

    if x is None:
        return []

    if isinstance(x, list):
        return [str(a) for a in x]

    if isinstance(x, str):
        s = x.strip()

        if s.startswith("[") and s.endswith("]"):
            try:
                v = json.loads(s)
                if isinstance(v, list):
                    return [str(a) for a in v]
            except:
                pass

        return [s]

    return [str(x)]

def first_hit_rank(retrieved_docs, gold_aliases):

    if not retrieved_docs or not gold_aliases:
        return None

    gold_norm = [norm(g) for g in gold_aliases if norm(g)]

    for i, d in enumerate(retrieved_docs[:TOP_K], start=1):

        txt = d.get(TEXT_KEY, "") if isinstance(d, dict) else str(d)
        txt = norm(txt)

        for g in gold_norm:
            if g and g in txt:
                return i

    return None

def mean01(xs):
    return float(np.mean(xs)) if xs else 0.0


# ============================================================
# LOAD DATASET
# ============================================================

ds = load_dataset("MinaGabriel/entityquestions-retrieval20-with-sre")["train"]
n = len(ds)

print("Dataset size:", n)


# ============================================================
# METRICS
# ============================================================

metrics = {
    "ALL": {"hits": {k: [] for k in K_LIST}, "n": 0},
    "LONG-TAIL": {"hits": {k: [] for k in K_LIST}, "n": 0},
    "INFREQUENT": {"hits": {k: [] for k in K_LIST}, "n": 0},
    "FREQUENT": {"hits": {k: [] for k in K_LIST}, "n": 0},
}


# ============================================================
# EVALUATION LOOP
# ============================================================

for ex in tqdm(ds, desc="FUSE-QA Evaluation"):

    q = ex["question"]
    gold = parse_answers(ex.get(ANS_KEY))
    s_pop = int(ex.get("s_pop", 0))

    # ------------------------------------------------
    # FULL FUSE-QA PIPELINE
    # BM25 + DPR + RRF + BGE reranker
    # ------------------------------------------------

    retrieved = retrieve_and_rerank(
        q,
        k_bm25=K_BM25,
        k_dense=K_DENSE,
        fuse_k=FUSE_K,
        top_k=TOP_K
    )

    hit_rank = first_hit_rank(retrieved, gold)

    # ------------------------------------------------
    # Tier assignment
    # ------------------------------------------------

    if s_pop < LT_THRESHOLD:
        tier = "LONG-TAIL"
    elif s_pop < FREQ_THRESHOLD:
        tier = "INFREQUENT"
    else:
        tier = "FREQUENT"

    for group in ["ALL", tier]:

        metrics[group]["n"] += 1

        for k in K_LIST:

            val = 1 if (hit_rank is not None and hit_rank <= k) else 0
            metrics[group]["hits"][k].append(val)


# ============================================================
# REPORT GENERATION
# ============================================================

lines = []
lines.append("=" * 80)
lines.append("FUSE-QA Retrieval Report (BM25 + DPR + RRF + BGE)")
lines.append("=" * 80)
lines.append(f"Dataset: EntityQuestions test")
lines.append(f"n = {n}")
lines.append("")
lines.append("Tier Definitions:")
lines.append(f"LONG-TAIL   s_pop < {LT_THRESHOLD}")
lines.append(f"INFREQUENT  {LT_THRESHOLD} ≤ s_pop < {FREQ_THRESHOLD}")
lines.append(f"FREQUENT    s_pop ≥ {FREQ_THRESHOLD}")
lines.append("")

for name in ["ALL", "LONG-TAIL", "INFREQUENT", "FREQUENT"]:

    m = metrics[name]

    lines.append(f"{name:<12} (n={m['n']})")

    for k in K_LIST:
        score = mean01(m["hits"][k])
        lines.append(f"  R@{k:<2} = {score:.4f}")

    lines.append("")


with open(OUT_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("Saved report:", OUT_FILE)